# M04 — ML Algorithms from Scratch

**Why implement from scratch?** Because interviews ask you to explain *why* Ridge regression shrinks coefficients, *why* decision trees split on information gain, and *what* the kernel trick actually does. You can't explain what you haven't built.

**Format:** Each exercise has:
1. The mathematical derivation (pre-written — read it carefully)
2. A scaffold class to implement
3. An assertion that compares your result to sklearn within tolerance

**Allowed:** `numpy` only for implementations. `sklearn` only for verification.

**Reference:** Build everything from first principles using only numpy documentation.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing, fetch_openml, make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Datasets
housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]
X_reg = housing.drop('medhousval', axis=1).values
y_reg = housing['medhousval'].values

X_cls, y_cls = make_classification(
    n_samples=2000, n_features=10, n_informative=6,
    n_redundant=2, random_state=42
)

# Standard splits
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

scaler = StandardScaler()
X_tr_r_s = scaler.fit_transform(X_tr_r)
X_te_r_s = scaler.transform(X_te_r)
X_tr_c_s = StandardScaler().fit_transform(X_tr_c)
X_te_c_s = StandardScaler().fit_transform(X_te_c)

print(f"Regression: {X_tr_r.shape} | Classification: {X_tr_c.shape}")

---
## Exercise 1 — Linear Regression: OLS from Scratch

## The Math

We want to minimize the sum of squared residuals:
$$\mathcal{L}(\beta) = \|y - X\beta\|^2 = (y - X\beta)^T(y - X\beta)$$

Taking the derivative and setting to zero:
$$\frac{\partial \mathcal{L}}{\partial \beta} = -2X^T(y - X\beta) = 0$$

Solving for $\beta$:
$$X^TX\beta = X^Ty \implies \hat{\beta} = (X^TX)^{-1}X^Ty$$

This is the **Normal Equation**. It has a closed-form solution — no iteration needed.

**Why does this fail at scale?** Computing $(X^TX)^{-1}$ is $O(p^3)$ where $p$ is the number of features. For $p > 10{,}000$, gradient descent is faster.

**Task:** Implement `LinearRegressionScratch` using the Normal Equation. Add a bias term by appending a column of ones to X.

In [ ]:
class LinearRegressionScratch:
    """
    Ordinary Least Squares via the Normal Equation.
    beta = (X^T X)^{-1} X^T y
    """
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LinearRegressionScratch':
        """
        Fit using Normal Equation.
        Add bias column, solve for beta, separate intercept from coef.
        """
        # YOUR CODE HERE
        pass

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        y_hat = X @ coef_ + intercept_
        """
        # YOUR CODE HERE
        pass

lr_scratch = LinearRegressionScratch()
lr_scratch.fit(X_tr_r_s, y_tr_r)
preds_scratch = lr_scratch.predict(X_te_r_s)

In [ ]:
# --- ASSERTIONS vs sklearn ---
lr_sk = LinearRegression().fit(X_tr_r_s, y_tr_r)
preds_sk = lr_sk.predict(X_te_r_s)

assert lr_scratch.coef_ is not None
assert lr_scratch.intercept_ is not None
assert np.allclose(lr_scratch.coef_, lr_sk.coef_, atol=1e-4), \
    f"Coefficients don't match. Max diff: {np.abs(lr_scratch.coef_ - lr_sk.coef_).max():.6f}"
assert abs(lr_scratch.intercept_ - lr_sk.intercept_) < 1e-4
rmse_scratch = np.sqrt(mean_squared_error(y_te_r, preds_scratch))
rmse_sk = np.sqrt(mean_squared_error(y_te_r, preds_sk))
assert abs(rmse_scratch - rmse_sk) < 1e-4
print(f"✓ Exercise 1 passed — RMSE: {rmse_scratch:.4f} (sklearn: {rmse_sk:.4f})")

---
## Exercise 2 — Ridge Regression: Regularized Normal Equation

## The Math

Ridge adds an L2 penalty to the OLS objective:
$$\mathcal{L}(\beta) = \|y - X\beta\|^2 + \lambda\|\beta\|^2$$

Taking the derivative and setting to zero:
$$-2X^T(y - X\beta) + 2\lambda\beta = 0$$

Solving:
$$\hat{\beta}_{\text{Ridge}} = (X^TX + \lambda I)^{-1}X^Ty$$

**Why does this shrink coefficients?** The $\lambda I$ term increases the diagonal of $X^TX$, which reduces the magnitude of all $\beta$ values toward zero.

**Why doesn't the penalty apply to the intercept?** The intercept just shifts predictions up/down — penalizing it would bias predictions for off-center targets. We exclude it by not regularizing the bias column.

**Task:** Implement `RidgeRegressionScratch`. The key difference from OLS: add `lambda * I` to `X^TX` before inverting. **Do not regularize the intercept** — zero out that diagonal entry.

In [ ]:
class RidgeRegressionScratch:
    """
    Ridge: beta = (X^T X + lambda * I)^{-1} X^T y
    Do NOT regularize the intercept term.
    """
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'RidgeRegressionScratch':
        # YOUR CODE HERE
        pass

    def predict(self, X: np.ndarray) -> np.ndarray:
        # YOUR CODE HERE
        pass

# Test across multiple alpha values
ridge_scratch = RidgeRegressionScratch(alpha=10.0)
ridge_scratch.fit(X_tr_r_s, y_tr_r)

In [ ]:
# --- ASSERTIONS vs sklearn ---
ridge_sk = Ridge(alpha=10.0).fit(X_tr_r_s, y_tr_r)

assert np.allclose(ridge_scratch.coef_, ridge_sk.coef_, atol=1e-3), \
    f"Max coef diff: {np.abs(ridge_scratch.coef_ - ridge_sk.coef_).max():.6f}"
assert abs(ridge_scratch.intercept_ - ridge_sk.intercept_) < 1e-3

# Verify shrinkage: Ridge coefs must be smaller in magnitude than OLS
assert np.abs(ridge_scratch.coef_).max() < np.abs(lr_scratch.coef_).max(), \
    "Ridge coefs must be shrunk vs OLS"

# Verify: larger alpha → more shrinkage
ridge_big = RidgeRegressionScratch(alpha=1000.0)
ridge_big.fit(X_tr_r_s, y_tr_r)
assert np.abs(ridge_big.coef_).max() < np.abs(ridge_scratch.coef_).max(), \
    "Larger alpha must produce more shrinkage"

print(f"✓ Exercise 2 passed")
print(f"OLS max |coef|: {np.abs(lr_scratch.coef_).max():.4f}")
print(f"Ridge(10) max |coef|: {np.abs(ridge_scratch.coef_).max():.4f}")
print(f"Ridge(1000) max |coef|: {np.abs(ridge_big.coef_).max():.4f}")

---
## Exercise 3 — Gradient Descent: The Iterative Alternative

## The Math

For large datasets, the Normal Equation is too slow. We use **gradient descent** instead:
$$\beta^{(t+1)} = \beta^{(t)} - \eta \nabla_{\beta} \mathcal{L}$$

For OLS, the gradient is:
$$\nabla_{\beta} \mathcal{L} = \frac{2}{n} X^T(X\beta - y)$$

Three variants:
- **Batch GD:** Use all $n$ samples per step. Slow per epoch, stable convergence.
- **Stochastic GD (SGD):** Use 1 sample per step. Fast but noisy.
- **Mini-batch GD:** Use $b$ samples per step. Best of both worlds.

**Convergence condition:** Learning rate $\eta < \frac{2}{\lambda_{\max}(X^TX/n)}$ where $\lambda_{\max}$ is the largest eigenvalue. In practice: try values like 0.01, 0.1 and watch the loss curve.

**Task:** Implement `GradientDescentRegressor` with all three variants. Track loss per iteration.

In [ ]:
class GradientDescentRegressor:
    """
    Linear regression via gradient descent.
    Supports: 'batch', 'sgd', 'minibatch'
    """
    def __init__(self, lr: float = 0.01, n_epochs: int = 100,
                 batch_size: int = 32, method: str = 'batch',
                 random_state: int = 42):
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.method = method
        self.random_state = random_state
        self.coef_ = None
        self.intercept_ = None
        self.loss_history_ = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'GradientDescentRegressor':
        """
        Fit using the chosen gradient descent variant.
        Store loss per epoch in self.loss_history_.
        """
        # YOUR CODE HERE
        # 1. Initialize weights to zeros
        # 2. Add bias column to X
        # 3. Loop epochs:
        #    - If batch: use all data
        #    - If sgd: shuffle and use 1 sample at a time
        #    - If minibatch: shuffle and use batch_size samples
        #    - Compute gradient: (2/n) * X_batch.T @ (X_batch @ beta - y_batch)
        #    - Update: beta -= lr * gradient
        #    - Record MSE loss for the epoch
        pass

    def predict(self, X: np.ndarray) -> np.ndarray:
        # YOUR CODE HERE
        pass

gd_batch = GradientDescentRegressor(lr=0.1, n_epochs=200, method='batch')
gd_batch.fit(X_tr_r_s, y_tr_r)

gd_sgd = GradientDescentRegressor(lr=0.01, n_epochs=50, method='sgd')
gd_sgd.fit(X_tr_r_s, y_tr_r)

gd_mini = GradientDescentRegressor(lr=0.05, n_epochs=100, method='minibatch', batch_size=64)
gd_mini.fit(X_tr_r_s, y_tr_r)

In [ ]:
# --- ASSERTIONS ---
for model, name in [(gd_batch,'batch'), (gd_sgd,'sgd'), (gd_mini,'minibatch')]:
    assert model.coef_ is not None, f"{name} coef_ not set"
    assert len(model.loss_history_) > 0, f"{name} loss_history_ empty"
    preds = model.predict(X_te_r_s)
    rmse = np.sqrt(mean_squared_error(y_te_r, preds))
    print(f"{name}: RMSE={rmse:.4f}, final_loss={model.loss_history_[-1]:.4f}")

# Loss must decrease over training (for batch GD at least)
first_half = np.mean(gd_batch.loss_history_[:len(gd_batch.loss_history_)//2])
second_half = np.mean(gd_batch.loss_history_[len(gd_batch.loss_history_)//2:])
assert second_half < first_half, "Batch GD loss must decrease over epochs"

# Batch GD result must be close to Normal Equation
rmse_gd = np.sqrt(mean_squared_error(y_te_r, gd_batch.predict(X_te_r_s)))
rmse_ols = np.sqrt(mean_squared_error(y_te_r, preds_scratch))
assert abs(rmse_gd - rmse_ols) < 0.1, f"GD should converge close to OLS: {rmse_gd:.4f} vs {rmse_ols:.4f}"
print(f"✓ Exercise 3 passed")

---
## Exercise 4 — Logistic Regression from Scratch

## The Math

Logistic regression models the probability of class 1 using the sigmoid function:
$$P(y=1|x) = \sigma(x^T\beta) = \frac{1}{1 + e^{-x^T\beta}}$$

The loss is **binary cross-entropy** (negative log-likelihood):
$$\mathcal{L}(\beta) = -\frac{1}{n}\sum_{i=1}^n \left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right]$$

The gradient (the beautiful result from differentiation):
$$\nabla_{\beta} \mathcal{L} = \frac{1}{n} X^T(\hat{p} - y)$$

This is identical in form to the linear regression gradient — replace predictions with probabilities.

**Numerical stability:** Never compute $\log(0)$. Clip probabilities: $\hat{p} = \text{clip}(\hat{p}, \epsilon, 1-\epsilon)$ for small $\epsilon$.

**Task:** Implement `LogisticRegressionScratch` using gradient descent. Support L2 regularization.

In [ ]:
class LogisticRegressionScratch:
    """
    Binary logistic regression via gradient descent.
    Loss: binary cross-entropy + optional L2 regularization.
    """
    def __init__(self, lr: float = 0.1, n_epochs: int = 200,
                 C: float = 1.0, random_state: int = 42):
        self.lr = lr
        self.n_epochs = n_epochs
        self.C = C  # inverse regularization strength (like sklearn)
        self.random_state = random_state
        self.coef_ = None
        self.intercept_ = None
        self.loss_history_ = []

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        """Numerically stable sigmoid."""
        # YOUR CODE HERE
        # Hint: use np.where to avoid overflow for large negative z
        pass

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LogisticRegressionScratch':
        """
        Gradient descent on binary cross-entropy with L2 penalty.
        Gradient: (1/n) * X.T @ (p_hat - y) + (1/C) * beta (don't regularize intercept)
        """
        # YOUR CODE HERE
        pass

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Returns (n_samples, 2) array of [P(0), P(1)]."""
        # YOUR CODE HERE
        pass

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """Binary predictions at given threshold."""
        # YOUR CODE HERE
        pass

lr_scratch = LogisticRegressionScratch(lr=0.1, n_epochs=300, C=1.0)
lr_scratch.fit(X_tr_c_s, y_tr_c)

In [ ]:
# --- ASSERTIONS vs sklearn ---
lr_sk = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_sk.fit(X_tr_c_s, y_tr_c)

proba_scratch = lr_scratch.predict_proba(X_te_c_s)[:, 1]
proba_sk = lr_sk.predict_proba(X_te_c_s)[:, 1]

auc_scratch = roc_auc_score(y_te_c, proba_scratch)
auc_sk = roc_auc_score(y_te_c, proba_sk)

assert auc_scratch > 0.7, f"AUC too low: {auc_scratch:.4f}"
assert abs(auc_scratch - auc_sk) < 0.05, f"AUC should be close to sklearn: {auc_scratch:.4f} vs {auc_sk:.4f}"

# Loss must decrease
assert lr_scratch.loss_history_[-1] < lr_scratch.loss_history_[0]

# Sigmoid sanity check
assert abs(LogisticRegressionScratch._sigmoid(0) - 0.5) < 1e-10
assert LogisticRegressionScratch._sigmoid(100) > 0.999
assert LogisticRegressionScratch._sigmoid(-100) < 0.001

print(f"✓ Exercise 4 passed — AUC: {auc_scratch:.4f} (sklearn: {auc_sk:.4f})")

---
## Exercise 5 — Decision Tree from Scratch

## The Math

A decision tree splits data by choosing the feature and threshold that maximizes **information gain**:
$$\text{IG}(D, f, t) = H(D) - \frac{|D_L|}{|D|} H(D_L) - \frac{|D_R|}{|D|} H(D_R)$$

Where $H$ is **entropy** (for classification):
$$H(D) = -\sum_{k} p_k \log_2(p_k)$$

Or **Gini impurity** (faster, similar results):
$$G(D) = 1 - \sum_{k} p_k^2$$

**Stopping criteria:**
- Max depth reached
- Node has fewer than `min_samples_split` samples
- All samples in node have the same class (entropy = 0)

**Task:** Implement a binary decision tree classifier using Gini impurity. Support `max_depth` and `min_samples_split`.

In [ ]:
class DecisionNode:
    """A node in the decision tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None,
                 value=None, n_samples=None, gini=None):
        self.feature = feature        # Feature index to split on
        self.threshold = threshold    # Split threshold
        self.left = left              # Left subtree (feature <= threshold)
        self.right = right            # Right subtree (feature > threshold)
        self.value = value            # Predicted class (only for leaf nodes)
        self.n_samples = n_samples
        self.gini = gini

class DecisionTreeScratch:
    """
    Binary decision tree classifier using Gini impurity.
    """
    def __init__(self, max_depth: int = 5, min_samples_split: int = 2,
                 random_state: int = 42):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.root = None
        self.n_features_ = None
        self.classes_ = None

    def _gini(self, y: np.ndarray) -> float:
        """
        Gini impurity: 1 - sum(p_k^2)
        """
        # YOUR CODE HERE
        pass

    def _best_split(self, X: np.ndarray, y: np.ndarray):
        """
        Find the best (feature, threshold) pair by exhaustive search.
        For each feature: sort unique values, try midpoints as thresholds.
        Return (best_feature, best_threshold, best_gini_gain)
        """
        # YOUR CODE HERE
        pass

    def _build_tree(self, X: np.ndarray, y: np.ndarray, depth: int = 0) -> DecisionNode:
        """
        Recursively build the tree.
        Stop if: max_depth reached, n_samples < min_samples_split, or all samples same class.
        """
        # YOUR CODE HERE
        pass

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'DecisionTreeScratch':
        self.classes_ = np.unique(y)
        self.n_features_ = X.shape[1]
        self.root = self._build_tree(X, y)
        return self

    def _traverse(self, x: np.ndarray, node: DecisionNode) -> int:
        """Traverse tree for a single sample."""
        if node.value is not None:  # leaf node
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([self._traverse(x, self.root) for x in X])

dt_scratch = DecisionTreeScratch(max_depth=4, min_samples_split=5)
dt_scratch.fit(X_tr_c, y_tr_c)

In [ ]:
# --- ASSERTIONS vs sklearn ---
dt_sk = DecisionTreeClassifier(max_depth=4, min_samples_split=5, criterion='gini', random_state=42)
dt_sk.fit(X_tr_c, y_tr_c)

preds_scratch = dt_scratch.predict(X_te_c)
preds_sk = dt_sk.predict(X_te_c)

acc_scratch = accuracy_score(y_te_c, preds_scratch)
acc_sk = accuracy_score(y_te_c, preds_sk)

assert acc_scratch > 0.70, f"Accuracy too low: {acc_scratch:.4f}"
assert abs(acc_scratch - acc_sk) < 0.05, f"Accuracy gap too large: {acc_scratch:.4f} vs {acc_sk:.4f}"

# Gini test
assert dt_scratch._gini(np.array([0,0,0,0])) == 0.0, "Pure node: gini=0"
assert abs(dt_scratch._gini(np.array([0,1,0,1])) - 0.5) < 1e-10, "50/50 split: gini=0.5"

print(f"✓ Exercise 5 passed — Accuracy: {acc_scratch:.4f} (sklearn: {acc_sk:.4f})")

---
## Exercise 6 — K-Nearest Neighbors from Scratch

## The Math

KNN is a non-parametric method: it makes no assumptions about the data distribution. A prediction for point $x$ is based on the $k$ training points closest to it.

**Euclidean distance:** $d(x, x_i) = \sqrt{\sum_j (x_j - x_{ij})^2}$

**Classification:** majority vote among $k$ neighbors.

**Curse of dimensionality:** In high dimensions, all points are approximately equidistant. KNN degrades as $d$ grows because the notion of "nearest" becomes meaningless.

**Task:** Implement `KNNClassifierScratch`. Vectorize the distance computation using numpy broadcasting — no loops over training points.

In [ ]:
class KNNClassifierScratch:
    """
    K-Nearest Neighbors classifier.
    Vectorized: O(n_test * n_train * d) — no Python loops over training points.
    """
    def __init__(self, k: int = 5):
        self.k = k
        self.X_train_ = None
        self.y_train_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'KNNClassifierScratch':
        """Store training data."""
        # YOUR CODE HERE
        pass

    def _compute_distances(self, X_test: np.ndarray) -> np.ndarray:
        """
        Compute pairwise Euclidean distances without loops.
        Returns (n_test, n_train) distance matrix.
        Hint: ||a - b||^2 = ||a||^2 + ||b||^2 - 2 * a.T @ b
        """
        # YOUR CODE HERE
        pass

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        For each test point: find k nearest, return majority class.
        """
        # YOUR CODE HERE
        pass

# Use small subset for speed
X_tr_small = X_tr_c_s[:500]
y_tr_small = y_tr_c[:500]

knn = KNNClassifierScratch(k=7)
knn.fit(X_tr_small, y_tr_small)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_sk = KNeighborsClassifier(n_neighbors=7).fit(X_tr_small, y_tr_small)

preds_scratch = knn.predict(X_te_c_s)
preds_sk = knn_sk.predict(X_te_c_s)

acc_scratch = accuracy_score(y_te_c, preds_scratch)
acc_sk = accuracy_score(y_te_c, preds_sk)

assert acc_scratch > 0.60
# Results should be nearly identical
agreement = (preds_scratch == preds_sk).mean()
assert agreement > 0.95, f"Predictions agree {agreement:.2%} with sklearn (expected >95%)"

# Vectorization check: _compute_distances must return (n_test, n_train)
dists = knn._compute_distances(X_te_c_s[:10])
assert dists.shape == (10, len(X_tr_small))

print(f"✓ Exercise 6 passed — Accuracy: {acc_scratch:.4f} | Agreement with sklearn: {agreement:.2%}")

---
## Exercise 7 — Naive Bayes from Scratch

## The Math

**Bayes theorem:**
$$P(y|x) = \frac{P(x|y) \cdot P(y)}{P(x)}$$

**Naive assumption:** features are conditionally independent given $y$:
$$P(x|y) = \prod_{j=1}^d P(x_j|y)$$

**Gaussian Naive Bayes:** assumes each feature follows $\mathcal{N}(\mu_{jk}, \sigma_{jk}^2)$ for class $k$:
$$P(x_j|y=k) = \frac{1}{\sqrt{2\pi\sigma_{jk}^2}} \exp\left(-\frac{(x_j - \mu_{jk})^2}{2\sigma_{jk}^2}\right)$$

**Log-sum trick:** compute $\log P(y|x)$ to avoid numerical underflow from multiplying many small probabilities:
$$\log P(y|x) \propto \log P(y) + \sum_j \log P(x_j|y)$$

**Task:** Implement `GaussianNaiveBayesScratch` that estimates $\mu_{jk}$ and $\sigma_{jk}^2$ from training data.

In [ ]:
class GaussianNaiveBayesScratch:
    """
    Gaussian Naive Bayes classifier.
    Estimates mu and sigma^2 per (feature, class) from training data.
    Uses log-probabilities for numerical stability.
    """
    def __init__(self):
        self.class_priors_ = None    # log P(y=k) per class
        self.class_means_ = None     # shape (n_classes, n_features)
        self.class_vars_ = None      # shape (n_classes, n_features)
        self.classes_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'GaussianNaiveBayesScratch':
        """
        Compute per-class mean, variance, and log-prior.
        """
        # YOUR CODE HERE
        pass

    def _log_likelihood(self, X: np.ndarray) -> np.ndarray:
        """
        Compute log P(x|y=k) for each class k.
        Returns (n_samples, n_classes)
        """
        # YOUR CODE HERE
        # For each class: sum of log N(x_j; mu_jk, sigma_jk^2) over features j
        pass

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Returns (n_samples, n_classes) probability array.
        """
        # YOUR CODE HERE
        # log_posterior = log_prior + log_likelihood (per class)
        # Convert to probabilities using softmax
        pass

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

gnb = GaussianNaiveBayesScratch()
gnb.fit(X_tr_c, y_tr_c)

In [ ]:
from sklearn.naive_bayes import GaussianNB

gnb_sk = GaussianNB().fit(X_tr_c, y_tr_c)

proba_scratch = gnb.predict_proba(X_te_c)
proba_sk = gnb_sk.predict_proba(X_te_c)

assert proba_scratch.shape == (len(y_te_c), 2)
assert np.allclose(proba_scratch.sum(axis=1), 1.0, atol=1e-5), "Probs must sum to 1"

auc_scratch = roc_auc_score(y_te_c, proba_scratch[:, 1])
auc_sk = roc_auc_score(y_te_c, proba_sk[:, 1])
assert abs(auc_scratch - auc_sk) < 0.02

# Verify means match training data means per class
for k_idx, k in enumerate(gnb.classes_):
    expected_mean = X_tr_c[y_tr_c == k].mean(axis=0)
    assert np.allclose(gnb.class_means_[k_idx], expected_mean, atol=1e-6)

print(f"✓ Exercise 7 passed — AUC: {auc_scratch:.4f} (sklearn: {auc_sk:.4f})")

---
## Exercise 8 — Principal Component Analysis from Scratch

## The Math

PCA finds the directions of maximum variance in the data. The principal components are the **eigenvectors** of the covariance matrix, ordered by decreasing eigenvalue.

**Steps:**
1. Center the data: $\tilde{X} = X - \bar{X}$
2. Compute covariance matrix: $C = \frac{1}{n-1} \tilde{X}^T \tilde{X}$
3. Eigendecomposition: $C = V \Lambda V^T$ where columns of $V$ are eigenvectors
4. Sort by decreasing eigenvalue
5. Project: $Z = \tilde{X} V_{[:k]}$

**Equivalently via SVD** (more numerically stable): $\tilde{X} = U \Sigma V^T$, principal components = columns of $V$.

**Explained variance:** $\text{EVR}_i = \lambda_i / \sum_j \lambda_j$

**Task:** Implement both the covariance/eigendecomposition method and the SVD method. Verify they give the same result (up to sign).

In [ ]:
class PCAScratch:
    """
    PCA via eigendecomposition of the covariance matrix.
    """
    def __init__(self, n_components: int = 2, method: str = 'eig'):
        self.n_components = n_components
        self.method = method  # 'eig' or 'svd'
        self.components_ = None            # shape (n_components, n_features)
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None
        self.mean_ = None

    def fit(self, X: np.ndarray) -> 'PCAScratch':
        """
        Fit PCA using eigendecomposition or SVD.
        Store components_ sorted by decreasing explained variance.
        """
        # YOUR CODE HERE
        # 1. Center X: store mean
        # 2. If method='eig': compute cov matrix, np.linalg.eigh (symmetric)
        # 3. If method='svd': use np.linalg.svd(X_centered, full_matrices=False)
        # 4. Sort descending
        # 5. Store first n_components
        pass

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Project X onto principal components."""
        # YOUR CODE HERE
        pass

    def inverse_transform(self, Z: np.ndarray) -> np.ndarray:
        """Reconstruct X from projected data."""
        # YOUR CODE HERE
        pass

pca_eig = PCAScratch(n_components=3, method='eig')
pca_eig.fit(X_tr_c)

pca_svd = PCAScratch(n_components=3, method='svd')
pca_svd.fit(X_tr_c)

In [ ]:
from sklearn.decomposition import PCA

pca_sk = PCA(n_components=3)
pca_sk.fit(X_tr_c)

# Explained variance ratio must match sklearn
assert np.allclose(
    np.abs(pca_eig.explained_variance_ratio_),
    np.abs(pca_sk.explained_variance_ratio_),
    atol=1e-4
), "EVR mismatch (eig method)"

assert np.allclose(
    np.abs(pca_svd.explained_variance_ratio_),
    np.abs(pca_sk.explained_variance_ratio_),
    atol=1e-4
), "EVR mismatch (svd method)"

# Projection must match (up to sign flip)
Z_eig = pca_eig.transform(X_te_c)
Z_svd = pca_svd.transform(X_te_c)
Z_sk = pca_sk.transform(X_te_c)

for i in range(3):
    corr = np.corrcoef(Z_eig[:, i], Z_sk[:, i])[0, 1]
    assert abs(abs(corr) - 1.0) < 0.001, f"PC{i+1} projection doesn't match sklearn"

# Reconstruction error must be small
X_reconstructed = pca_eig.inverse_transform(Z_eig)
recon_error = np.mean((X_te_c - X_reconstructed) ** 2)
assert recon_error < np.var(X_te_c), "Reconstruction error must be less than total variance"

print(f"✓ Exercise 8 passed")
print(f"EVR: {pca_eig.explained_variance_ratio_.round(4)} (sum: {pca_eig.explained_variance_ratio_.sum():.4f})")

---
## Exercise 9 — K-Means from Scratch

## The Math

K-Means minimizes the **within-cluster sum of squares (WCSS)**:
$$\text{WCSS} = \sum_{k=1}^K \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

**Lloyd's algorithm:**
1. Initialize $K$ centroids (random or K-Means++ )
2. **Assignment step:** assign each point to nearest centroid
3. **Update step:** recompute centroids as mean of assigned points
4. Repeat until centroids don't change

**K-Means++ initialization:** Choose centroids with probability proportional to squared distance to nearest existing centroid. This dramatically improves convergence and final clustering.

**Task:** Implement K-Means with both random and K-Means++ initialization. Track inertia per iteration.

In [ ]:
class KMeansScratch:
    """
    K-Means clustering with random or K-Means++ initialization.
    """
    def __init__(self, k: int = 3, max_iter: int = 100,
                 init: str = 'kmeans++', random_state: int = 42):
        self.k = k
        self.max_iter = max_iter
        self.init = init  # 'random' or 'kmeans++'
        self.random_state = random_state
        self.centroids_ = None
        self.labels_ = None
        self.inertia_ = None
        self.inertia_history_ = []
        self.n_iter_ = 0

    def _init_centroids(self, X: np.ndarray) -> np.ndarray:
        """
        Initialize centroids.
        'random': pick k random points from X
        'kmeans++': first centroid random, each subsequent chosen with probability
                    proportional to squared distance to nearest existing centroid.
        """
        # YOUR CODE HERE
        pass

    def _assign(self, X: np.ndarray) -> np.ndarray:
        """
        Assign each point to nearest centroid.
        Returns labels array of shape (n_samples,).
        Vectorized: compute all distances at once.
        """
        # YOUR CODE HERE
        pass

    def _update(self, X: np.ndarray, labels: np.ndarray) -> np.ndarray:
        """
        Recompute centroids as mean of assigned points.
        Handle empty clusters: keep previous centroid.
        """
        # YOUR CODE HERE
        pass

    def fit(self, X: np.ndarray) -> 'KMeansScratch':
        np.random.seed(self.random_state)
        self.centroids_ = self._init_centroids(X)
        for i in range(self.max_iter):
            old_centroids = self.centroids_.copy()
            self.labels_ = self._assign(X)
            self.centroids_ = self._update(X, self.labels_)
            self.inertia_ = float(np.sum((X - self.centroids_[self.labels_]) ** 2))
            self.inertia_history_.append(self.inertia_)
            self.n_iter_ = i + 1
            if np.allclose(old_centroids, self.centroids_):
                break
        return self

km = KMeansScratch(k=3, init='kmeans++', random_state=42)
km.fit(X_tr_c)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

km_sk = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=1)
km_sk.fit(X_tr_c)

assert km.centroids_ is not None and km.centroids_.shape == (3, X_tr_c.shape[1])
assert len(km.labels_) == len(X_tr_c)
assert len(set(km.labels_)) == 3, "Must have 3 clusters"

# Inertia must decrease monotonically
assert all(km.inertia_history_[i] >= km.inertia_history_[i+1]
           for i in range(len(km.inertia_history_)-1)), "Inertia must not increase"

# Our inertia should be close to sklearn's
assert abs(km.inertia_ - km_sk.inertia_) / km_sk.inertia_ < 0.1, \
    f"Inertia too different: {km.inertia_:.1f} vs {km_sk.inertia_:.1f}"

sil = silhouette_score(X_tr_c, km.labels_)
print(f"✓ Exercise 9 passed — Inertia: {km.inertia_:.1f} | Silhouette: {sil:.4f} | Iters: {km.n_iter_}")

---
## Exercise 10 — Capstone: Implement and Compare All Algorithms

**Spec:** Build a unified benchmark that runs all scratch implementations against sklearn on the same dataset, and measures:

1. For each classification algorithm (LogReg, DecisionTree, KNN, NaiveBayes): accuracy, AUC, training time, prediction time.
2. For PCA: explained variance ratio for 1–5 components, reconstruction error.
3. For K-Means: silhouette score, inertia, convergence iterations.
4. Return `benchmark_df`: rows = (algorithm, implementation), columns = all metrics.

**Most important question to answer:** For which algorithms is your scratch implementation closest to sklearn? Where is there the biggest gap, and why?

In [ ]:
import time

# YOUR CODE HERE
benchmark_df = None

In [ ]:
# --- ASSERTIONS ---
assert benchmark_df is not None
assert len(benchmark_df) >= 8  # 4 algorithms x 2 implementations
assert 'algorithm' in benchmark_df.columns or benchmark_df.index.name == 'algorithm'
print("✓ Exercise 10 passed")
print(benchmark_df.to_string())

**Analysis:** *(For which algorithm is your scratch implementation closest to sklearn? Where is the biggest gap? What would make it closer?)*